In [10]:
import os
from dataclasses import dataclass

from dotenv import load_dotenv

load_dotenv()


@dataclass(frozen=True)
class Provider:
    """One provider described as pure DATA (same design as Notebook 1)."""

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str


PROVIDERS = [
    Provider("OpenAI",     "OPENAI_API_KEY",     False, None,                              "gpt-4o-mini"),
    Provider("Groq",       "GROQ_API_KEY",       True,  "https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.environ.get(provider.env_var):
            return provider
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set. Add one of {expected} to your .env file.")


def build_client(provider: Provider):
    from openai import OpenAI

    api_key = os.environ[provider.env_var]
    if provider.base_url is None:
        return OpenAI(api_key=api_key)
    return OpenAI(api_key=api_key, base_url=provider.base_url)


def have_any_key() -> bool:
    return any(os.environ.get(p.env_var) for p in PROVIDERS)



def llm_reply(prompt: str, *, max_tokens: int = 400, temperature: float | None = None) -> str:
    """Send one user prompt; return the assistant's text."""
    provider = select_provider()
    client = build_client(provider)
    request = {
        "model": provider.model,
        "max_tokens": max_tokens,
        "messages": [{"role": "user", "content": prompt}],
    }
    if temperature is not None:
        request["temperature"] = temperature
    result = client.chat.completions.create(**request)
    return result.choices[0].message.content

In [11]:
import re


def cot_prompt(question: str) -> str:
    """Ask for step-by-step reasoning and a machine-readable numeric answer."""
    return f"""Solve this problem by thinking step by step.

Question: {question}

End your response with exactly: Final answer: <number>"""


def extract_final_number(text: str) -> int | float:
    """Return the number after the final `Final answer:` label."""
    matches = re.findall(
        r"(?im)^\s*Final answer:\s*([-+]?(?:\d+(?:\.\d*)?|\.\d+)(?:[eE][-+]?\d+)?)\s*$",
        text,
    )
    if not matches:
        raise ValueError("No numeric `Final answer:` found in model output.")

    number = matches[-1]
    return float(number) if any(char in number.lower() for char in (".", "e")) else int(number)


def sample_cot_paths(
    question: str, *, n_samples: int = 5, temperature: float = 0.7
) -> list[str]:
    """Generate independent CoT responses for one question."""
    if n_samples < 1:
        raise ValueError("n_samples must be at least 1.")
    prompt = cot_prompt(question)
    return [
        llm_reply(prompt, temperature=temperature)
        for _ in range(n_samples)
    ]


def run_self_consistency(
    question: str, *, n_samples: int = 5, temperature: float = 0.7
) -> "VoteResult":
    """Sample independent reasoning paths and return their majority vote."""
    reasoning_samples = sample_cot_paths(
        question, n_samples=n_samples, temperature=temperature
    )
    return self_consistent_answer(reasoning_samples)

## Self-consistency voting

Several reasoning paths can be sampled for the same GSM8K problem. We extract each final answer and choose the number with the most votes.

The self-consistency research paper uses 40 samples. We use 5 here because every sample requires a separate API call and therefore adds cost and latency.

In [12]:
from collections import Counter
from dataclasses import dataclass


Number = int | float


@dataclass(frozen=True)
class VoteResult:
    """The outcome of voting over independently generated answers."""

    winner: Number
    tally: dict[Number, int]
    samples: int
    confidence: float


def majority_vote(answers: list[Number]) -> VoteResult:
    """Return the most-voted answer along with the complete vote tally."""
    if not answers:
        raise ValueError("At least one answer is required to vote.")

    tally = Counter(answers)
    winner, winning_votes = tally.most_common(1)[0]
    samples = len(answers)
    return VoteResult(
        winner=winner,
        tally=dict(tally),
        samples=samples,
        confidence=winning_votes / samples,
    )


def self_consistent_answer(reasoning_samples: list[str]) -> VoteResult:
    """Extract final numbers from model responses and choose the majority answer."""
    answers = [extract_final_number(sample) for sample in reasoning_samples]
    return majority_vote(answers)


GSM8K_QUESTION = (
    "Janet's ducks lay 16 eggs per day. She eats 3 eggs for breakfast every morning "
    "and bakes muffins for her friends every day with 4 eggs. She sells the rest "
    "for $2 per egg. How much does she make every day?"
)

# These stand in for five independently sampled model reasoning paths.
FAKE_MODEL_ANSWERS = [
    "Janet has 16 eggs, uses 3 for breakfast and 4 for muffins, leaving 9. At $2 each, 9 * 2 = 18.\nFinal answer: 18",
    "After breakfast and baking, the eggs left are 16 - 3 - 4 = 9. Selling them earns 9 times 2 dollars.\nFinal answer: 18",
    "She sells 16 - 3 - 4 = 9 eggs. Her earnings are 9 x $2 = $18.\nFinal answer: 18",
    "The 4 muffin eggs are included in the 3 breakfast eggs, so 16 - 3 = 13 eggs remain. 13 * 2 = 26.\nFinal answer: 26",
    "There are 9 eggs left after using 3 + 4 = 7 eggs. Multiplying 9 by 2 gives 18.\nFinal answer: 18",
]

result = self_consistent_answer(FAKE_MODEL_ANSWERS)

print(GSM8K_QUESTION)
print(f"Vote tally: {result.tally}")
print(f"Self-consistency winner: {result.winner}")
print(f"Confidence: {result.confidence:.0%} ({result.samples} samples)")

Janet's ducks lay 16 eggs per day. She eats 3 eggs for breakfast every morning and bakes muffins for her friends every day with 4 eggs. She sells the rest for $2 per egg. How much does she make every day?
Vote tally: {18: 4, 26: 1}
Self-consistency winner: 18
Confidence: 80% (5 samples)


## Greedy decoding vs. self-consistency

Greedy decoding takes one path at temperature 0. Self-consistency samples five paths at temperature 0.7, then selects the most common extracted answer.

In [13]:
if not have_any_key():
    print("Set OPENAI_API_KEY or GROQ_API_KEY to run the live comparison.")
else:
    # One greedy path: deterministic decoding at temperature 0.
    greedy_reply = llm_reply(cot_prompt(GSM8K_QUESTION), temperature=0.0)
    print("=== One call (greedy, temperature=0) ===")
    print(greedy_reply)
    print(f"Extracted number: {extract_final_number(greedy_reply)}")

    # Five independent paths: stochastic decoding followed by majority voting.
    reasoning_samples = sample_cot_paths(
        GSM8K_QUESTION, n_samples=5, temperature=0.7
    )
    result = self_consistent_answer(reasoning_samples)
    print("\n=== Self-consistency (5 calls, temperature=0.7) ===")
    for index, (reply, answer) in enumerate(
        zip(reasoning_samples, [extract_final_number(s) for s in reasoning_samples]),
        start=1,
    ):
        print(f"\n--- Sample {index} (answer: {answer}) ---")
        print(reply)
    print(f"\nVote tally: {result.tally}")
    print(f"Winner: {result.winner}")
    print(f"Confidence: {result.confidence:.0%}")

=== One call (greedy, temperature=0) ===
To solve the problem step by step, we need to determine how many eggs Janet has left after she eats and bakes with them, and then calculate how much money she makes from selling the remaining eggs.

1. **Total eggs laid by ducks per day**: Janet's ducks lay 16 eggs per day.

2. **Eggs eaten for breakfast**: Janet eats 3 eggs for breakfast every morning.

3. **Eggs used for baking muffins**: Janet uses 4 eggs to bake muffins for her friends.

4. **Total eggs consumed (eaten + baked)**: 
   \[
   \text{Total eggs consumed} = \text{Eggs for breakfast} + \text{Eggs for baking} = 3 + 4 = 7 \text{ eggs}
   \]

5. **Eggs left after consumption**: 
   \[
   \text{Eggs left} = \text{Total eggs laid} - \text{Total eggs consumed} = 16 - 7 = 9 \text{ eggs}
   \]

6. **Selling price per egg**: Janet sells the remaining eggs for $2 per egg.

7. **Total money made from selling eggs**: 
   \[
   \text{Total money made} = \text{Eggs left} \times \text{Selling pr